# 01b — Supplementary Public Sources (ITU / UK Parliament / CEPT / NCC)

Fetches and merges supplementary regulatory and telecom statistics:

- **ITU DataHub API** — fixed telephone, broadband, internet users
- **UK House of Commons Library** — PSTN / digital landline switch-off timeline
- **CEPT ECC Report 265** — European PSTN/ISDN → IP migration references
- **NCC Taiwan (via data.gov.tw / api.ncc.gov.tw)** — mobile subscribers and landline number allocation

Outputs: enriched `data/processed/panel_data.csv`, raw caches under `data/raw/`, and `data/processed/regulatory_sources_summary.json`.

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.cwd()))

from src.data.fetcher import load_config, fetch_all
from src.data.preprocessor import (
    add_derived_features,
    add_pstn_phaseout_feature,
    align_panel,
    impute_missing,
    save_processed,
)
from src.data.supplementary_fetcher import (
    merge_pstn_switchoff_dates,
    ncc_to_annual_panel,
)

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)

In [ ]:
config = load_config('config.yaml')
cache_dir = Path('data/raw')
cache_dir.mkdir(parents=True, exist_ok=True)

all_data = fetch_all(config, force_refetch=False)

itu = all_data['itu']
berec_dates = all_data['berec_dates']
uk = all_data['uk_parliament']
cept = all_data['cept']
ncc = all_data['ncc']

merged_switchoff = merge_pstn_switchoff_dates(config, berec_dates, uk, cept)
print(f'ITU rows: {len(itu)}')
print(f"UK Parliament fetch status: {uk['fetch_status']} -> {uk.get('switchoff_year')}")
print(f"CEPT fetch status: {cept['fetch_status']} -> {len(cept.get('countries', []))} country mentions")
print(f'NCC Taiwan rows: {len(ncc)}')
print(f"Merged PSTN switch-off countries with dates: {sum(v is not None for v in merged_switchoff.values())}")

In [ ]:
panel_path = Path('data/processed/panel_data.csv')
if panel_path.exists():
    panel = pd.read_csv(panel_path)
else:
    wb = all_data['world_bank']
    panel = align_panel(
        {'world_bank': wb},
        countries=wb['country'].unique().tolist(),
        year_range=range(2000, 2026),
    )
    panel = impute_missing(panel, method='linear', max_gap=3)

drop_cols = [c for c in panel.columns if c.startswith(('itu_', 'ncc_'))]
panel = panel.drop(columns=drop_cols, errors='ignore')

extra_sources = {'itu': itu, 'ncc_annual': ncc_to_annual_panel(ncc)}
for _, frame in extra_sources.items():
    if frame.empty:
        continue
    panel = panel.merge(frame, on=['country', 'year'], how='left')

panel = add_pstn_phaseout_feature(panel, merged_switchoff)
panel = add_derived_features(panel)
save_processed(panel)

summary = {
    'itu_indicator_columns': [c for c in panel.columns if c.startswith('itu_')],
    'ncc_columns': [c for c in panel.columns if c.startswith('ncc_')],
    'uk_parliament': uk,
    'cept': cept,
    'merged_pstn_switchoff': merged_switchoff,
}
Path('data/processed/regulatory_sources_summary.json').write_text(
    json.dumps(summary, ensure_ascii=False, indent=2) + '\n'
)

print(f'Updated panel shape: {panel.shape}')
print(panel.filter(regex='^(itu_|ncc_)', axis=1).head())

---
### Summary

In [ ]:
print('Supplementary columns in panel:')
print([c for c in panel.columns if c.startswith(('itu_', 'ncc_'))])
print('\nTaiwan NCC sample:')
print(ncc.head(10).to_string(index=False) if not ncc.empty else 'No NCC rows fetched')
print('\nUK excerpt:')
print(uk.get('excerpt', '')[:400])